Function that will be used to crawl the publication sites and acquire a list of unique publications along with other info

In [ ]:
import requests
from bs4 import BeautifulSoup as bsSoup
import time

In [ ]:
# scrape the links for all the publications of https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm
publicationsIndex = "https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm"

def getLinks (url: str) -> list :
    sitePrefix = "https://sitescrape.awh.durham.ac.uk/comp42315/"

    if (not isinstance(url, str)) :
        print("The URL needs to be a string!")
        return []

    # wait a little so we don't overload the server
    time.sleep(2)
    publicationsIndex = requests.get(url, verify = False)

    if (publicationsIndex.status_code != 200) :
        print (f"Error; status code returned: {publicationsIndex.status_code}")
        return []

    publicationsIndexSoup = bsSoup(publicationsIndex.content, "html.parser").body

    if (publicationsIndexSoup == None) :
        print ("Nothin to parse on the site")
        return []

    publicationLinksA = publicationsIndexSoup.find("p", class_ = "TextOption").find_all("a")

    if (publicationLinksA == None) :
        print ("No links found")
        return []
    
    links = [sitePrefix + n.get("href") for n in publicationLinksA]

    if (links[0] == None) :
        print("No links found")
        return []

    return links

publicationLinks = [publicationsIndex] + getLinks(publicationsIndex)

def scrapePublications (urls: list) -> list:
    uniquePublications = set()

    for url in urls :
        content = []

        time.sleep(2)
        publicationSite = requests.get(url, verify = False)
        
        if (publicationSite.status_code != 200) :
            print(f"Failed for link {url}, status code: {publicationSite.status_code}, continuing execution for the remaining links")
            continue

        publicationSiteSoup = bsSoup(publicationSite.content, "html.parser").body

        if (publicationSiteSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        publicationsDiv = publicationSiteSoup.find_all("div", style = "margin-left: var(--size-marginleft)")

        if (len(publicationsDiv) == 0) :
            print (f"Couldn't find publications on site {url}, continuing execution for the remaining links")
            continue

        content = [n.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle").text.strip() for n in publicationsDiv]

    # compare the titles somewhere
    # find first by? and this is a cutoff for the title
    return content 

def scrapeImpactScores (urls: list) -> dict :
    impactScores = {}
    # if a site has been seen, ignore it
    seenSites = set()
    #n = 0
    
    for url in urls :
        #n += 1
        #print(n)
        uniqueScoresOnThePage = []
        # change that 0 to 1 later
        time.sleep(0)
        url = url.replace("year", "impactfactor")
        
        scoreSite = requests.get(url, verify = False)

        if (scoreSite.status_code != 200) :
            print(f"Failed for link {url}, status code: {scoreSite.status_code}, continuing execution for the remaining links")
            continue

        scoreSiteSoup = bsSoup(scoreSite.content, "html.parser").body

        if (scoreSiteSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        scoreSiteSoup = scoreSiteSoup.find("div", id = "divBackground")
        
        if (scoreSiteSoup == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        scoreSiteSoupP = scoreSiteSoup.find_all("p", class_ = "TextOption")

        if (scoreSiteSoupP == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        paragraphWithScores = scoreSiteSoupP [2]

        uniqueScoresOnThePage = [n.text for n in paragraphWithScores.find_all("a")]

        if (len(uniqueScoresOnThePage) == 0) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        # for each link in unique links find the publication titles that correspond
        for score in uniqueScoresOnThePage :
            currentH2 = scoreSiteSoup.find("h2", id = score)

            if (currentH2 == None) :
                # no publications under this score
                continue

            if (score not in impactScores) :
                impactScores [score] = []

            div = currentH2.findNext()

            publicationsWithThisScore = div.find_all("div", class_ = "w3-cell-row")

            if (publicationsWithThisScore == None) :
                # no publications under this score
                continue

            for publication in publicationsWithThisScore :
                publicationText = publication.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle")

                # I'd need to find the title here
                title = publicationText.text.split("by")[0].strip()

                if (title in seenSites) :
                    continue

                seenSites.add(title)
                impactScores[score].append(publicationText.text)
                #print(title)

    return impactScores
                                                      
#print (publicationLinks)
#print(scrapePublications(publicationLinks))   
scoreDict = scrapeImpactScores(publicationLinks)
print(scoreDict)

This information should then be stored and displayed in an appropriate format. Displayed to the user should be: publication title, publication venue, year and author list of every unique publication record on the target website.

In [ ]:
# extract info from score dict - dictionary{title: <>, value [list of the characteristics]}
publications = {}

for k, v in scoreDict.items() :
    titleSplitIndex = v.find("by")
    authorSplitIndex = 